In [1]:
from typing import List
import pandas as pd
from pathlib import Path

def structured_merge_gui_files(file_paths: List[Path], output_path: Path) -> pd.DataFrame:
    """
    Merges multiple *_gui.csv files into a structured long-form file with:
    - One ENG row per verse_id (full columns intact)
    - One LAN row per verse_id per language (no change, since inputs are clean)

    Assumes each file has:
    - lang == ENG row for each verse_id
    - lang == [LANG] row for each verse_id
    - Identical structure across files

    Args:
        file_paths (List[Path]): List of *_gui.csv files to merge
        output_path (Path): Path to save the merged CSV
    Returns:
        pd.DataFrame: The structured merged DataFrame
    """
    data_by_lang = {}
    verse_id_sets = []

    # Load each file and organize by language
    for path in file_paths:
        df = pd.read_csv(path)
        langs = df['lang'].unique()
        assert len(langs) == 2, f"File {path.name} must contain exactly one target language + ENG."
        lang_code = [l for l in langs if l != 'ENG'][0]
        data_by_lang[lang_code] = df
        verse_id_sets.append(set(df['verse_id']))

    # Only use verse_ids that are shared across all files
    common_ids = set.intersection(*verse_id_sets)

    if not common_ids:
        raise ValueError("No overlapping verse_ids found across files!")

    sorted_ids = sorted(common_ids)

    merged_rows = []

    for vid in sorted_ids:
        # Add the English row from any file
        eng_row = None
        for df in data_by_lang.values():
            row = df[(df['verse_id'] == vid) & (df['lang'] == 'ENG')]
            if not row.empty:
                eng_row = row.iloc[0].to_dict()
                break
        if eng_row is None:
            raise ValueError(f"No English row found for verse_id {vid}")
        merged_rows.append(eng_row)

        # Add each language's row
        for lang, df in data_by_lang.items():
            lan_row = df[(df['verse_id'] == vid) & (df['lang'] == lang)]
            if lan_row.empty:
                raise ValueError(f"Missing {lang} row for verse_id {vid}")
            merged_rows.append(lan_row.iloc[0].to_dict())

    final_df = pd.DataFrame(merged_rows)
    final_df.to_csv(output_path, index=False)
    return final_df

In [2]:
from pathlib import Path

file_paths = [
    Path("deu_gui.csv"),
    Path("xho_gui.csv"),
    Path("yor_gui.csv"),
]

output_path = Path("prelim_gui_input.csv")

merged_df = structured_merge_gui_files(file_paths, output_path)
merged_df.head()

,verse_id,anchor,lang,text,constr,metadata,akt,exp,res,univ,dtam,evid,tam_lg,neg,ant
0,1001031,0,ENG,God saw all that he had made . It was very goo...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1001031,1,DEU,"und/ gott/GOD sah/SEE alles/ALL an/ ,/ was/THA...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1001031,1,XHO,wakubona/SEE uthixo/GOD konke/ALL akwenzileyo/...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1001031,1,YOR,ọlọrun/GOD si/ ri/SEE ohun/ gbogbo/ALL ti/THAT...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1002002,0,ENG,By the seventh day God had finished his work ....,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
